# LeetCode 2078 — Two Furthest Houses With Different Colors bằng Python

Link đề: https://leetcode.com/problems/two-furthest-houses-with-different-colors/ · Độ khó: **Easy** · Chủ đề: Array, Greedy

```python
class Solution:
    def maxDistance(self, colors: List[int]) -> int:

```

**Mục tiêu học**

1. Viết brute force **thử mọi cặp** `i < j` (vòng lặp lồng nhau) và hiểu chi phí `O(n²)`.
2. Rút ra **nhận xét tham lam (greedy)**: cặp xa nhất luôn chứa **nhà đầu** hoặc **nhà cuối** — và chứng minh được điều đó.
3. Cài đặt `O(n)` bằng hai lần quét từ hai đầu; biết chỉ số âm `colors[-1]`, `range` đếm lùi, `max`.
4. Phân biệt các cách viết sai phổ biến (chỉ xét một đầu, so sánh nhầm phần tử).
5. Nộp `class Solution`, đối chiếu brute force trên test ngẫu nhiên.

## 1. Đề bài yêu cầu gì?

`n` ngôi nhà xếp thẳng hàng, `colors[i]` là màu nhà `i` (đánh số từ 0). Trả về **khoảng cách lớn nhất** giữa hai nhà **khác màu**. Khoảng cách giữa nhà `i` và `j` là `abs(i - j)`.

| Ví dụ | colors | Output | Cặp tốt nhất |
|---|---|---|---|
| 1 | `[1,1,1,6,1,1,1]` | 3 | (0, 3) hoặc (3, 6) |
| 2 | `[1,8,3,8,3]` | 4 | (0, 4): màu 1 và 3 |
| 3 | `[0,1]` | 1 | (0, 1) |

Constraints: `2 <= n <= 100`, `0 <= colors[i] <= 100`, **đảm bảo có ít nhất hai nhà khác màu**.

Chú ý: khoảng cách tính theo **chỉ số** (vị trí), không theo giá trị màu.

In [1]:
colors = [1, 8, 3, 8, 3]
i, j = 0, 4
print(f"nhà {i} màu {colors[i]}, nhà {j} màu {colors[j]}, khác màu? {colors[i] != colors[j]}, khoảng cách {abs(i - j)}")

nhà 0 màu 1, nhà 4 màu 3, khác màu? True, khoảng cách 4


## 2. Nền tảng Python

### 2.1. Vòng lặp lồng nhau để duyệt mọi cặp `i < j`

`for i in range(n): for j in range(i + 1, n):` liệt kê mỗi cặp **đúng một lần** (không lặp `(j, i)`, không có `i == j`). Số cặp: `n(n−1)/2`.

In [2]:
n = 4
pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
print(pairs, len(pairs), "=", n * (n - 1) // 2)

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)] 6 = 6


### 2.2. `abs`, `max` và cập nhật "giá trị tốt nhất"

Mẫu quen thuộc: `best = 0`, mỗi khi tìm được ứng viên tốt hơn thì `best = max(best, candidate)`.

In [3]:
best = 0
for candidate in [2, 5, 3]:
    best = max(best, candidate)
    print("ứng viên", candidate, "-> best =", best)
print(abs(1 - 4), abs(4 - 1))

ứng viên 2 -> best = 2
ứng viên 5 -> best = 5
ứng viên 3 -> best = 5
3 3


### 2.3. Chỉ số âm và `range` đếm lùi

- `colors[-1]` là phần tử **cuối**, `colors[-2]` là áp cuối.
- `range(n - 1, -1, -1)` cho `n−1, n−2, ..., 0` (dừng **trước** `-1`).

In [4]:
colors = [1, 8, 3, 8, 3]
print(colors[-1], colors[len(colors) - 1])
print(list(range(len(colors) - 1, -1, -1)))

3 3
[4, 3, 2, 1, 0]


### 2.4. `break` — thoát vòng lặp sớm

Khi quét từ cuối về để tìm nhà **xa nhất** khác màu nhà 0, nhà đầu tiên gặp được **là** xa nhất → `break` ngay.

In [5]:
for j in range(4, -1, -1):
    print("xét j =", j)
    if j == 2:
        print("gặp điều kiện -> break")
        break

xét j = 4
xét j = 3
xét j = 2
gặp điều kiện -> break


## 3. Cách 1 — Brute force: thử mọi cặp

Với mọi `i < j`, nếu `colors[i] != colors[j]` thì cập nhật `best = max(best, j - i)` (vì `i < j` nên `abs(i - j) = j - i`).

In [6]:
def max_distance_bruteforce(colors):
    n = len(colors)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            if colors[i] != colors[j]:
                best = max(best, j - i)
    return best

print(max_distance_bruteforce([1, 1, 1, 6, 1, 1, 1]))   # 3
print(max_distance_bruteforce([1, 8, 3, 8, 3]))         # 4
print(max_distance_bruteforce([0, 1]))                  # 1

3
4
1


Với `n ≤ 100` (≈ 5.000 cặp) brute force đã qua LeetCode. Nhưng đây là bài "Greedy" — ta muốn hiểu **vì sao** không cần xét mọi cặp.

### Cải tiến nhỏ trước: duyệt theo khoảng cách giảm dần

Thử khoảng cách `d = n−1, n−2, ...`; với mỗi `d` thử mọi cặp `(i, i + d)`. Cặp khác màu đầu tiên tìm thấy cho đáp án. Vẫn `O(n²)` tệ nhất, nhưng gợi ý điều quan trọng: với `d` lớn, chỉ có **ít** cặp, và chúng luôn dính sát **hai đầu** mảng.

In [7]:
def max_distance_by_d(colors):
    n = len(colors)
    for d in range(n - 1, 0, -1):
        for i in range(0, n - d):
            if colors[i] != colors[i + d]:
                return d
    return 0

print(max_distance_by_d([1, 1, 1, 6, 1, 1, 1]), max_distance_by_d([1, 8, 3, 8, 3]), max_distance_by_d([0, 1]))
for d in [6, 5, 4]:
    print(f"d={d}: các cặp", [(i, i + d) for i in range(7 - d)])

3 4 1
d=6: các cặp [(0, 6)]
d=5: các cặp [(0, 5), (1, 6)]
d=4: các cặp [(0, 4), (1, 5), (2, 6)]


## 4. Insight — cặp xa nhất luôn chứa nhà 0 hoặc nhà n−1

**Khẳng định:** tồn tại một cặp tối ưu có `i = 0` hoặc `j = n − 1`.

**Chứng minh (đổi một đầu mút):** giả sử cặp tối ưu `(i, j)`, `i < j`, `colors[i] != colors[j]`, và `0 < i`, `j < n − 1`. Xét nhà `0`:

- Nếu `colors[0] != colors[j]` → cặp `(0, j)` khác màu và **xa hơn** (`j − 0 > j − i`). Mâu thuẫn với "tối ưu".
- Nếu `colors[0] == colors[j]` → xét nhà `n−1`:
  - Nếu `colors[n−1] != colors[i]` → cặp `(i, n−1)` khác màu và xa hơn `(i, j)` (vì `n−1 > j`). Mâu thuẫn.
  - Nếu `colors[n−1] == colors[i]` → khi đó `colors[n−1] = colors[i] != colors[j] = colors[0]`, nên cặp `(0, n−1)` khác màu với khoảng cách `n−1` — lớn nhất có thể, xa hơn `(i, j)`. Mâu thuẫn.

Mọi trường hợp đều mâu thuẫn, nên cặp tối ưu phải có `i = 0` hoặc `j = n − 1`. ∎

**Hệ quả:** đáp án = `max(` xa nhất từ nhà 0 tới nhà khác màu nhà 0 `,` xa nhất từ nhà n−1 tới nhà khác màu nhà n−1 `)`. Đúng như gợi ý thứ hai của đề.

In [8]:
# Kiểm chứng khẳng định bằng thực nghiệm: luôn có cặp tối ưu chứa 0 hoặc n-1
import random
random.seed(0)
for _ in range(3000):
    n = random.randint(2, 9)
    colors = [random.randint(0, 2) for _ in range(n)]
    if len(set(colors)) < 2:
        continue
    best = max_distance_bruteforce(colors)
    pairs = [(i, j) for i in range(n) for j in range(i + 1, n)
             if colors[i] != colors[j] and j - i == best]
    assert any(i == 0 or j == n - 1 for i, j in pairs), colors
print("Mọi bộ thử đều có cặp tối ưu chạm một đầu mảng")

Mọi bộ thử đều có cặp tối ưu chạm một đầu mảng


## 5. Cách 2 (lời giải chính) — Hai lần quét từ hai đầu

1. **Từ nhà 0:** quét `j` từ `n−1` lùi về; nhà đầu tiên có `colors[j] != colors[0]` cho khoảng cách `j`.
2. **Từ nhà n−1:** quét `i` từ `0` tiến lên; nhà đầu tiên có `colors[i] != colors[n−1]` cho khoảng cách `n−1−i`.
3. Trả `max` của hai giá trị.

In [9]:
def max_distance_greedy(colors):
    n = len(colors)
    d1 = 0
    for j in range(n - 1, -1, -1):          # xa nhất tính từ nhà 0
        if colors[j] != colors[0]:
            d1 = j
            break
    d2 = 0
    for i in range(n):                       # xa nhất tính từ nhà n-1
        if colors[i] != colors[n - 1]:
            d2 = n - 1 - i
            break
    return max(d1, d2)

print(max_distance_greedy([1, 1, 1, 6, 1, 1, 1]))
print(max_distance_greedy([1, 8, 3, 8, 3]))
print(max_distance_greedy([0, 1]))

3
4
1


## 6. TRACE từng bước

Ví dụ 1: `colors = [1,1,1,6,1,1,1]`, `n = 7`.

**Quét 1** (so với `colors[0] = 1`, `j` từ 6 lùi):

| j | colors[j] | khác 1? | hành động |
|---|---|---|---|
| 6 | 1 | ✗ | tiếp |
| 5 | 1 | ✗ | tiếp |
| 4 | 1 | ✗ | tiếp |
| 3 | 6 | ✓ | `d1 = 3`, break |

**Quét 2** (so với `colors[6] = 1`, `i` từ 0 tiến): `i = 3` là nhà đầu tiên khác màu → `d2 = 6 − 3 = 3`.

Đáp án `max(3, 3) = 3`.

In [10]:
def trace_greedy(colors):
    n = len(colors)
    print(f"colors = {colors}, n = {n}")
    d1 = 0
    print(f"Quét 1: tìm j xa nhất có màu khác colors[0] = {colors[0]}")
    for j in range(n - 1, -1, -1):
        diff = colors[j] != colors[0]
        print(f"  j={j}: màu {colors[j]} -> {'KHÁC' if diff else 'giống'}")
        if diff:
            d1 = j
            break
    d2 = 0
    print(f"Quét 2: tìm i xa nhất có màu khác colors[{n - 1}] = {colors[-1]}")
    for i in range(n):
        diff = colors[i] != colors[-1]
        print(f"  i={i}: màu {colors[i]} -> {'KHÁC' if diff else 'giống'}")
        if diff:
            d2 = n - 1 - i
            break
    print(f"d1 = {d1}, d2 = {d2}, đáp án = {max(d1, d2)}")
    return max(d1, d2)

trace_greedy([1, 1, 1, 6, 1, 1, 1])

colors = [1, 1, 1, 6, 1, 1, 1], n = 7
Quét 1: tìm j xa nhất có màu khác colors[0] = 1
  j=6: màu 1 -> giống
  j=5: màu 1 -> giống
  j=4: màu 1 -> giống
  j=3: màu 6 -> KHÁC
Quét 2: tìm i xa nhất có màu khác colors[6] = 1
  i=0: màu 1 -> giống
  i=1: màu 1 -> giống
  i=2: màu 1 -> giống
  i=3: màu 6 -> KHÁC
d1 = 3, d2 = 3, đáp án = 3


3

In [11]:
trace_greedy([1, 8, 3, 8, 3])

colors = [1, 8, 3, 8, 3], n = 5
Quét 1: tìm j xa nhất có màu khác colors[0] = 1
  j=4: màu 3 -> KHÁC
Quét 2: tìm i xa nhất có màu khác colors[4] = 3
  i=0: màu 1 -> KHÁC
d1 = 4, d2 = 4, đáp án = 4


4

Ở ví dụ 2, quét 1 dừng ngay tại `j = 4` (màu 3 ≠ 1) → `d1 = 4` = khoảng cách tối đa có thể (`n − 1`).

## 7. Cách 3 — Một vòng lặp duy nhất

Gộp hai quét vào một vòng duyệt mọi `i`:

- Nếu `colors[i] != colors[0]` → ứng viên `i` (cặp `(0, i)`).
- Nếu `colors[i] != colors[n−1]` → ứng viên `n−1−i` (cặp `(i, n−1)`).

Không dừng sớm nhưng code ngắn, dễ nhớ.

In [12]:
def max_distance_one_loop(colors):
    n = len(colors)
    best = 0
    for i, c in enumerate(colors):
        if c != colors[0]:
            best = max(best, i)
        if c != colors[-1]:
            best = max(best, n - 1 - i)
    return best

print(max_distance_one_loop([1, 1, 1, 6, 1, 1, 1]), max_distance_one_loop([1, 8, 3, 8, 3]), max_distance_one_loop([0, 1]))

3 4 1


## 8. Cách 4 — Đặc biệt nhanh khi hai đầu khác màu

Nếu `colors[0] != colors[n−1]` thì đáp án **ngay lập tức** là `n − 1` (khoảng cách lớn nhất có thể). Chỉ khi hai đầu cùng màu `c` ta mới cần tìm: nhà khác màu `c` **đầu tiên** (từ trái, vị trí `L`) và **cuối cùng** (từ phải, vị trí `R`); đáp án `max(R, n − 1 − L)`.

In [13]:
def max_distance_ends(colors):
    n = len(colors)
    if colors[0] != colors[-1]:
        return n - 1
    c = colors[0]
    L = next(i for i in range(n) if colors[i] != c)               # đề đảm bảo tồn tại
    R = next(j for j in range(n - 1, -1, -1) if colors[j] != c)
    return max(R, n - 1 - L)

print(max_distance_ends([1, 1, 1, 6, 1, 1, 1]), max_distance_ends([1, 8, 3, 8, 3]), max_distance_ends([0, 1]))

3 4 1


`next(generator)` không có `default` sẽ báo `StopIteration` nếu không tìm thấy — an toàn ở đây vì đề đảm bảo có ít nhất hai màu.

## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def maxDistance(self, colors: List[int]) -> int:
        n = len(colors)
        j = n - 1
        while colors[j] == colors[0]:      # xa nhất khác màu nhà 0
            j -= 1
        i = 0
        while colors[i] == colors[n - 1]:  # xa nhất khác màu nhà n-1
            i += 1
        return max(j, n - 1 - i)

sol = Solution()
print(sol.maxDistance([1, 1, 1, 6, 1, 1, 1]), sol.maxDistance([1, 8, 3, 8, 3]), sol.maxDistance([0, 1]))

3 4 1


Bản này dùng `while`: vòng `while colors[j] == colors[0]` chắc chắn dừng vì đề đảm bảo tồn tại nhà khác màu nhà 0 (nếu mọi nhà cùng màu nhà 0 thì không có hai nhà khác màu). Tương tự cho vòng thứ hai.

## 10. Vì sao đúng?

1. Theo **khẳng định ở mục 4**, đáp án tối ưu đạt được bởi một cặp chứa nhà `0` hoặc nhà `n−1`.
2. Trong các cặp chứa nhà `0`, cặp xa nhất là `(0, j*)` với `j*` = chỉ số **lớn nhất** có `colors[j*] != colors[0]`. Vòng quét lùi từ `n−1` dừng ở chỉ số lớn nhất như vậy (**invariant**: mọi `j` đã bỏ qua đều cùng màu nhà 0).
3. Tương tự, cặp xa nhất chứa nhà `n−1` là `(i*, n−1)` với `i*` nhỏ nhất có `colors[i*] != colors[n−1]`.
4. `max` của hai giá trị là tốt nhất trong các ứng viên — và theo (1) cũng là tốt nhất toàn cục.

## 11. Độ phức tạp

Gọi `n = len(colors)` (≤ 100).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force mọi cặp | `O(n²)` | `O(1)` | oracle |
| 1'. Thử d giảm dần | `O(n²)` tệ nhất | `O(1)` | gợi ý insight |
| 2. Hai lần quét + break | `O(n)` | `O(1)` | lời giải chuẩn |
| 3. Một vòng lặp | `O(n)` (luôn duyệt hết) | `O(1)` | ngắn |
| 4. Kiểm tra hai đầu trước | `O(1)` nếu hai đầu khác màu, `O(n)` tệ nhất | `O(1)` | |

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS ({actual})")

cases = [
    ("ví dụ 1", [1, 1, 1, 6, 1, 1, 1], 3),
    ("ví dụ 2", [1, 8, 3, 8, 3], 4),
    ("ví dụ 3 (n = 2)", [0, 1], 1),
    ("hai đầu khác màu", [4, 4, 4, 9], 3),
    ("khác màu duy nhất ở đầu", [2, 7, 7, 7, 7], 4),
    ("khác màu sát đầu trái", [5, 6, 5, 5, 5, 5], 4),
    ("khác màu sát đầu phải", [5, 5, 5, 5, 6, 5], 4),
    ("khác màu ngay giữa, n chẵn", [3, 3, 8, 3], 2),
    ("chỉ màu 0 và 100", [0, 0, 100, 0, 0], 2),
]
fs = [max_distance_bruteforce, max_distance_by_d, max_distance_greedy,
      max_distance_one_loop, max_distance_ends, Solution().maxDistance]
for name, colors, exp in cases:
    for f in fs:
        assert f(colors) == exp, (f, colors, f(colors))
    check(name, sol.maxDistance(colors), exp)

ví dụ 1                              PASS (3)
ví dụ 2                              PASS (4)
ví dụ 3 (n = 2)                      PASS (1)
hai đầu khác màu                     PASS (3)
khác màu duy nhất ở đầu              PASS (4)
khác màu sát đầu trái                PASS (4)
khác màu sát đầu phải                PASS (4)
khác màu ngay giữa, n chẵn           PASS (2)
chỉ màu 0 và 100                     PASS (2)


In [16]:
random.seed(2078)
count = 0
for _ in range(5000):
    n = random.randint(2, 12)
    colors = [random.randint(0, 3) for _ in range(n)]
    if len(set(colors)) < 2:
        continue                       # đề đảm bảo có hai màu khác nhau
    answers = {f(colors) for f in fs}
    assert len(answers) == 1, (colors, [f(colors) for f in fs])
    count += 1
print(f"{count} mảng ngẫu nhiên: 6 cài đặt khớp nhau — PASS")

4832 mảng ngẫu nhiên: 6 cài đặt khớp nhau — PASS


Ca `[5,6,5,5,5,5]`: nhà 1 (màu 6) với nhà 5 → khoảng cách 4. Nhà 0 với nhà 1 chỉ là 1 — cho thấy **không chỉ** xét cặp chứa nhà 0.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ xét cặp chứa nhà 0

In [17]:
def wrong_only_left(colors):
    best = 0
    for j in range(len(colors)):
        if colors[j] != colors[0]:
            best = max(best, j)
    return best

print("Sai :", wrong_only_left([5, 6, 5, 5, 5, 5]))     # 1
print("Đúng:", max_distance_greedy([5, 6, 5, 5, 5, 5]))  # 4

Sai : 1
Đúng: 4


### Lỗi 2: So sánh hai nhà **cạnh nhau** rồi "kéo dài"

Ý tưởng "tìm chỗ đổi màu đầu tiên rồi lấy n−1" sai khi chỗ đổi màu nằm giữa và hai đầu cùng màu.

### Lỗi 3: Tính khoảng cách theo giá trị màu

`abs(colors[i] - colors[j])` thay vì `abs(i - j)` — nhầm "giá trị" với "vị trí", lỗi giống Two Sum (trả value thay vì index).

In [18]:
def wrong_color_value(colors):
    best = 0
    n = len(colors)
    for i in range(n):
        for j in range(i + 1, n):
            if colors[i] != colors[j]:
                best = max(best, abs(colors[i] - colors[j]))
    return best

print("Sai :", wrong_color_value([1, 8, 3, 8, 3]))      # 7 (màu 1 và 8)
print("Đúng:", max_distance_greedy([1, 8, 3, 8, 3]))     # 4

Sai : 7
Đúng: 4


### Lỗi 4: Quét lùi mà quên `break`

Không `break` thì `d1` bị ghi đè bởi các `j` **nhỏ hơn** → mất giá trị xa nhất.

In [19]:
def wrong_no_break(colors):
    n = len(colors)
    d1 = 0
    for j in range(n - 1, -1, -1):
        if colors[j] != colors[0]:
            d1 = j                     # không break -> cuối cùng giữ j NHỎ nhất
    d2 = 0
    for i in range(n):
        if colors[i] != colors[-1]:
            d2 = n - 1 - i             # không break -> giữ i LỚN nhất
    return max(d1, d2)

print("Sai :", wrong_no_break([1, 8, 3, 8, 3]))         # lấy nhầm j/i gần
print("Đúng:", max_distance_greedy([1, 8, 3, 8, 3]))

Sai : 1
Đúng: 4


### Lỗi 5: `range(n - 1, 0, -1)` bỏ sót chỉ số 0

Ở quét 1 không sao (nhà 0 luôn cùng màu chính nó), nhưng thói quen này gây lỗi ở bài khác. Muốn bao gồm 0 thì `stop = -1`.

## 14. Mẹo nhận dạng pattern

- "**Khoảng cách lớn nhất** giữa hai phần tử thỏa điều kiện" → nghĩ tới **hai đầu mảng**: phần tử xa nhau nhất thường dính đầu mút.
- Lập luận "**đổi đầu mút**" (exchange): nếu cặp tối ưu không chạm đầu, chứng minh đổi một đầu sang biên không làm tệ hơn.
- Constraints nhỏ (n ≤ 100) → brute force `O(n²)` luôn là phương án dự phòng và oracle kiểm thử.
- Bài cùng họ: 1855 Maximum Distance Between a Pair of Values (hai con trỏ), 962 Maximum Width Ramp, 121 Best Time to Buy and Sell Stock (giữ cực trị từ một đầu), 2016 Maximum Difference Between Increasing Elements.

## 15. Biến thể

1. **Khoảng cách nhỏ nhất** giữa hai nhà khác màu: luôn là 1 nếu có hai nhà kề nhau khác màu — và luôn có! (vì có ít nhất hai màu, ở đâu đó màu phải đổi giữa hai nhà kề nhau).
2. **Trả về cặp chỉ số** `(i, j)` thay vì khoảng cách.
3. **Khoảng cách lớn nhất giữa hai nhà cùng màu:** với mỗi màu, lấy `vị_trí_cuối − vị_trí_đầu` (dùng dict lưu lần đầu xuất hiện).

In [20]:
def max_same_color_distance(colors):
    first = {}
    best = 0
    for i, c in enumerate(colors):
        if c not in first:
            first[c] = i                 # lần đầu gặp màu c
        best = max(best, i - first[c])
    return best

def min_distance_diff_color(colors):
    return 1 if any(colors[i] != colors[i + 1] for i in range(len(colors) - 1)) else -1

print(max_same_color_distance([1, 8, 3, 8, 3]))     # 8: 1..3 -> 2 ; 3: 2..4 -> 2
print(min_distance_diff_color([1, 1, 1, 6, 1, 1, 1]))

2
1


## 16. Bài tập tự luyện

**Bài 1 — Tính tay:** `colors = [2, 2, 5, 2, 2, 2, 7, 2]`. Thực hiện hai lần quét như mục 6, ghi `d1`, `d2`, đáp án.

**Bài 2 — Đoán output:** `list(range(3, -1, -1))`, `[4, 5, 6][-1]`, `max(2, 7)`, số cặp `i < j` khi `n = 6`.

**Bài 3 — TODO:** viết `exercise_max_distance(colors)` bằng hai lần quét (dùng `for` + `break`).

**Bài 4 — TODO:** `furthest_pair(colors)` trả về **cặp** `(i, j)` (`i < j`) đạt khoảng cách lớn nhất; ưu tiên cặp chứa nhà 0 nếu hòa.

**Bài 5 — TODO biến thể:** `max_distance_at_least_gap(colors, g)` — khoảng cách lớn nhất giữa hai nhà có `abs(colors[i] - colors[j]) >= g` (trả `0` nếu không có). Gợi ý: lập luận đầu mút có còn đúng không? Dùng brute force nếu chưa chắc.

**Bài 6 — Mini challenge:** chứng minh hoặc bác bỏ: "với bài 5, cặp tối ưu cũng luôn chứa nhà 0 hoặc nhà n−1". Viết code tìm **phản ví dụ** bằng thử ngẫu nhiên (`find_counterexample()`).

In [21]:
def exercise_max_distance(colors):
    # TODO 1: quét j từ n-1 lùi, tìm màu khác colors[0]
    # TODO 2: quét i từ 0 tiến, tìm màu khác colors[n-1]
    # TODO 3: return max(...)
    pass

# assert exercise_max_distance([1,8,3,8,3]) == 4

In [22]:
def furthest_pair(colors):
    # TODO
    pass

# assert furthest_pair([1,1,1,6,1,1,1]) == (0, 3)

In [23]:
def max_distance_at_least_gap(colors, g):
    # TODO
    pass

# assert max_distance_at_least_gap([1, 5, 2, 9], 4) == 3

In [24]:
def find_counterexample():
    # TODO: thử ngẫu nhiên, so brute force với "chỉ xét cặp chứa 0 hoặc n-1"
    pass

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | hai đầu cùng màu 2 | quét 1 dừng ở j = 6 | quét 2 dừng ở i = 2 → d2 = 5; đáp án 6 |
| 2 | stop không được lấy | chỉ số âm từ cuối | `[3,2,1,0]`, `6`, `7`, `15` |
| 3 | mục 5 | `for j in range(n-1, -1, -1)` | `if colors[j] != colors[0]: d1 = j; break` |
| 4 | giữ cả j và i | so sánh `j` với `n-1-i` | `return (0, j) if j >= n-1-i else (i, n-1)` |
| 5 | điều kiện mới vẫn đối xứng | thử brute force trước | `if abs(colors[i]-colors[j]) >= g` |
| 6 | điều kiện "khác nhau ≥ g" không có tính bắc cầu như "khác màu" | so `best` toàn cục với `best` chỉ từ hai đầu | in phản ví dụ đầu tiên |

## 18. Đáp án tham khảo

**Bài 1:** quét 1 (so với màu 2): `j = 7` màu 2 ✗, `j = 6` màu 7 ✓ → `d1 = 6`. Quét 2 (so với `colors[7] = 2`): `i = 0, 1` ✗, `i = 2` màu 5 ✓ → `d2 = 7 − 2 = 5`. Đáp án **6**.

**Bài 2:** `[3, 2, 1, 0]` · `6` · `7` · `15`.

**Bài 6:** điều kiện `abs(a − b) >= g` **không** có tính chất "nếu `a ≠ c` và `b = c` thì `a ≠ b`" mà chứng minh ở mục 4 dựa vào, nên khẳng định có thể sai. Ví dụ `colors = [5, 0, 10, 5]`, `g = 10`: chỉ cặp `(1, 2)` thỏa (khoảng cách 1), không chạm đầu mảng nào.

In [25]:
trace_greedy([2, 2, 5, 2, 2, 2, 7, 2])
print(list(range(3, -1, -1)), [4, 5, 6][-1], max(2, 7), 6 * 5 // 2)

colors = [2, 2, 5, 2, 2, 2, 7, 2], n = 8
Quét 1: tìm j xa nhất có màu khác colors[0] = 2
  j=7: màu 2 -> giống
  j=6: màu 7 -> KHÁC
Quét 2: tìm i xa nhất có màu khác colors[7] = 2
  i=0: màu 2 -> giống
  i=1: màu 2 -> giống
  i=2: màu 5 -> KHÁC
d1 = 6, d2 = 5, đáp án = 6
[3, 2, 1, 0] 6 7 15


In [26]:
def exercise_max_distance_answer(colors):
    n = len(colors)
    d1 = d2 = 0
    for j in range(n - 1, -1, -1):
        if colors[j] != colors[0]:
            d1 = j
            break
    for i in range(n):
        if colors[i] != colors[n - 1]:
            d2 = n - 1 - i
            break
    return max(d1, d2)

def furthest_pair_answer(colors):
    n = len(colors)
    j = n - 1
    while colors[j] == colors[0]:
        j -= 1
    i = 0
    while colors[i] == colors[n - 1]:
        i += 1
    return (0, j) if j >= n - 1 - i else (i, n - 1)

def max_distance_at_least_gap_answer(colors, g):
    n, best = len(colors), 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(colors[i] - colors[j]) >= g:
                best = max(best, j - i)
    return best

def find_counterexample_answer(seed=1):
    rng = random.Random(seed)              # bộ sinh ngẫu nhiên riêng, có seed
    for _ in range(10000):
        n = rng.randint(2, 6)
        colors = [rng.randint(0, 10) for _ in range(n)]
        g = rng.randint(1, 10)
        best = max_distance_at_least_gap_answer(colors, g)
        ends_only = max([j for j in range(n) if abs(colors[j] - colors[0]) >= g] +
                        [n - 1 - i for i in range(n) if abs(colors[i] - colors[-1]) >= g] + [0])
        if best != ends_only:
            return colors, g, best, ends_only
    return None

assert exercise_max_distance_answer([1, 8, 3, 8, 3]) == 4
assert furthest_pair_answer([1, 1, 1, 6, 1, 1, 1]) == (0, 3)
assert max_distance_at_least_gap_answer([1, 5, 2, 9], 4) == 3
assert max_distance_at_least_gap_answer([5, 0, 10, 5], 10) == 1
print("Phản ví dụ tìm được:", find_counterexample_answer())
random.seed(12)
for _ in range(1000):
    colors = [random.randint(0, 3) for _ in range(random.randint(2, 10))]
    if len(set(colors)) < 2:
        continue
    assert exercise_max_distance_answer(colors) == max_distance_bruteforce(colors)
    i, j = furthest_pair_answer(colors)
    assert colors[i] != colors[j] and j - i == max_distance_bruteforce(colors)
print("Tất cả đáp án bài tập — PASS")

Phản ví dụ tìm được: ([7, 1, 4, 3, 9, 8], 8, 3, 0)
Tất cả đáp án bài tập — PASS


Bài học từ bài 6: một mẹo greedy đúng cho bài này **không tự động đúng** cho bài "na ná". Luôn kiểm lại chứng minh — hoặc ít nhất là đối chiếu với brute force.

## 19. Tổng kết — template + checklist

**Template "khoảng cách xa nhất giữa hai phần tử khác nhau":**

```python
n = len(a)
j = n - 1
while a[j] == a[0]: j -= 1          # xa nhất khác a[0]
i = 0
while a[i] == a[n - 1]: i += 1      # xa nhất khác a[n-1]
return max(j, n - 1 - i)
```

**Checklist submit**

- [ ] Xét **cả hai** đầu (nhà 0 **và** nhà n−1).
- [ ] Khoảng cách theo **chỉ số**, không theo giá trị màu.
- [ ] Quét có `break` (hoặc `while`) để giữ vị trí **xa nhất**.
- [ ] Dựa vào đảm bảo "có ít nhất hai màu" để vòng `while` chắc chắn dừng.
- [ ] Tên method `maxDistance`, trả `int`.